# CalibrateQwen 00: setup, secrets, and data
We validate Hub access and materialize the two conversation datasets used by the training notebooks.

In [ ]:
from pathlib import Path

REPO_ROOT = Path('/content/AutoRegressive-Bhasha')
if not REPO_ROOT.exists():
    !git clone https://github.com/ritwikraha/AutoRegressive-Bhasha.git /content/AutoRegressive-Bhasha
%cd /content/AutoRegressive-Bhasha/calibrate_qwen
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, login

tinker_key = (userdata.get('TINKER_API_KEY') or '').strip()
hf_token = (userdata.get('HF_TOKEN') or '').strip()
if not tinker_key or not hf_token:
    raise ValueError('TINKER_API_KEY and HF_TOKEN must be nonempty Colab secrets with notebook access enabled.')
os.environ['TINKER_API_KEY'] = tinker_key
os.environ['HF_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)
identity = api.whoami()
repo = api.repo_info('ritwikraha/calibrate-qwen-curated', repo_type='dataset')
try:
    os.environ['WANDB_API_KEY'] = (userdata.get('WANDB_API_KEY') or '').strip()
except Exception:
    pass
print(f"Authenticated with Hugging Face as {identity['name']}; private dataset access verified.")

In [ ]:
from datasets import load_dataset

REPO_ID = 'ritwikraha/calibrate-qwen-curated'
base = load_dataset(REPO_ID, 'calibrated_mcq', token=hf_token)
phase1 = load_dataset(REPO_ID, 'phase1_2000', token=hf_token)
teacher = load_dataset(REPO_ID, 'teacher_phase1', token=hf_token)
base, phase1, teacher

In [ ]:
from training.prepare_training_data import prepare_training_file

hard_manifest = prepare_training_file(
    output_path='artifacts/training/hard_label_numeric.jsonl',
    repo_id=REPO_ID,
    variant='hard_label',
    confidence_format='numeric',
    hf_token=hf_token,
)
teacher_manifest = prepare_training_file(
    output_path='artifacts/training/teacher_numeric.jsonl',
    repo_id=REPO_ID,
    variant='teacher',
    confidence_format='numeric',
    abstention_threshold=0.6,
    hf_token=hf_token,
)
hard_manifest, teacher_manifest